# Is everything in Adelaide 20 minutes away?

The drive from every Greater Adelaide mesh block to eight places, at the speed limit with no traffic (`free`), in
TomTom's 2025 morning and evening rush-hour slowdown (`am`, `pm`), and by public transport arriving 8 to 9 am (`pt`).
Made in [Plotlay](https://www.getplotlay.com). This notebook reads only `../data`.

**Read it carefully.** The rush hour is a city-wide average slowdown, not a traffic forecast; the trips go to the
nearest place of each kind and leave out parking.

In [1]:
import pandas as pd
DATA = '../data'
mb = pd.read_csv(f'{DATA}/mesh_block_minutes.csv', dtype={'mb_code': str, 'sa2_code': str})
PLACES = ['cbd', 'beach', 'airport', 'hospital', 'uni', 'oval', 'hills', 'shops']
total = mb.residents.sum()
print(f'{len(mb):,} mesh blocks, {total:,.0f} residents')

15,797 mesh blocks, 1,386,882 residents


## Within 20 minutes of each place

A mesh block counts when its minutes are 20 or fewer; empty means not reached.

In [2]:
def share(run, place, limit=20):
    m = mb[f'{run}_{place}']
    return (mb.residents * (m <= limit)).sum() / total
table = pd.DataFrame({run: {p: share(run, p) for p in PLACES} for run in ['free', 'am', 'pm', 'pt']})
(table * 100).round(1).sort_values('free', ascending=False)

,free,am,pm,pt
shops,99.4,97.6,97.5,37.4
hospital,93.5,79.1,77.0,4.7
uni,92.3,64.9,64.3,5.0
beach,61.9,40.9,40.2,7.1
oval,51.3,23.9,22.2,1.2
cbd,48.3,21.5,20.0,2.1
airport,39.6,15.1,14.1,0.3
hills,24.6,8.6,8.4,0.2


## All eight: the strict reading

In [3]:
def all_eight(run, limit=20):
    ok = pd.concat([mb[f'{run}_{p}'] <= limit for p in PLACES], axis=1).all(axis=1)
    return (mb.residents * ok).sum() / total
for run in ['free', 'am', 'pm', 'typical', 'pt']:
    print(f'{run:8s} {all_eight(run) * 100:5.1f}% within 20 minutes of all eight')

free      10.1% within 20 minutes of all eight
am         0.0% within 20 minutes of all eight
pm         0.0% within 20 minutes of all eight
typical   28.7% within 20 minutes of all eight
pt         0.0% within 20 minutes of all eight


## How many of the eight: the headline is half or more (4 of 8)

In [4]:
def counts(run): return sum((mb[f'{run}_{p}'] <= 20).astype(int) for p in PLACES)
dist = pd.DataFrame({run: mb.groupby(counts(run)).residents.sum().reindex(range(9), fill_value=0) / total * 100 for run in ['free', 'am', 'pt']})
dist.round(1)

,free,am,pt
0,0.3,1.4,53.8
1,0.7,4.7,38.2
2,3.9,19.5,6.0
3,23.4,31.2,0.8
4,17.5,19.7,0.5
5,9.7,12.1,0.6
6,8.2,4.6,0.0
7,26.2,6.8,0.0
8,10.1,0.0,0.0


In [5]:
for run in ['free', 'am', 'pm', 'typical', 'pt']:
    c = counts(run)
    o = c.sort_values(); w = mb.residents[o.index].cumsum(); median = o[w >= total / 2].iloc[0]
    print(f"{run:8s} 3+ {(mb.residents * (c >= 3)).sum() / total * 100:5.1f}%   4+ {(mb.residents * (c >= 4)).sum() / total * 100:5.1f}%   5+ {(mb.residents * (c >= 5)).sum() / total * 100:5.1f}%   median {median} of 8")

free     3+  95.2%   4+  71.8%   5+  54.2%   median 5 of 8
am       3+  74.3%   4+  43.1%   5+  23.5%   median 3 of 8
pm       3+  72.5%   4+  42.2%   5+  21.2%   median 3 of 8
typical  3+  96.6%   4+  82.4%   5+  71.3%   median 7 of 8
pt       3+   2.0%   4+   1.2%   5+   0.6%   median 0 of 8


## Where it holds: areas where most residents have half the places or more

In [6]:
sa2 = pd.read_csv(f'{DATA}/sa2.csv', dtype={'sa2_code': str})
big = sa2[sa2.residents >= 3000]
print((big.share_half_free >= 0.5).sum(), 'of', len(big), 'areas hold at the speed limit;', (big.share_half_am >= 0.5).sum(), 'in the rush hour')
big.sort_values(['share_half_free', 'places_within_20_free'], ascending=False)[['sa2_name', 'residents', 'share_half_free', 'share_half_am', 'places_within_20_free']].head(10)

71 of 102 areas hold at the speed limit; 42 in the rush hour


,sa2_name,residents,share_half_free,share_half_am,places_within_20_free
0,Goodwood - Millswood,17632,1.0,1.000,8.00
1,Unley - Parkside,20956,1.0,1.000,8.00
2,Colonel Light Gardens,15862,1.0,0.995,8.00
3,Mitcham (SA),16473,1.0,0.563,7.99
4,Adelaide,18223,1.0,1.000,7.91
5,Toorak Gardens,16635,1.0,1.000,7.67
6,Walkerville,7998,1.0,1.000,7.60
7,Beaumont - Glen Osmond,10320,1.0,0.790,7.57
8,Norwood (SA),10854,1.0,1.000,7.55
9,Edwardstown,14552,1.0,1.000,7.52


## Where it fails

In [7]:
big.sort_values(['share_half_free', 'places_within_20_free'])[['sa2_name', 'residents', 'share_half_free', 'places_within_20_free', 'mean_min_city_free']].head(10)

,sa2_name,residents,share_half_free,places_within_20_free,mean_min_city_free
108,Adelaide Hills,7032,0.0,1.01,39.5
107,Mount Barker Surrounds,6851,0.0,1.56,41.6
106,Lobethal - Woodside,9410,0.0,1.76,34.3
105,Lewiston - Two Wells,6480,0.0,1.86,37.0
104,Nairne,5332,0.0,2.00,33.6
103,Mount Barker,22080,0.0,2.12,31.5
102,Hahndorf - Echunga,4740,0.0,2.59,30.2
100,Gawler - North,9521,0.0,2.70,39.5
99,Gawler - South,21679,0.0,2.79,40.1
98,Willunga,3615,0.0,2.82,46.1
